In [1]:
#!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118 
#!pip install pycocotools tensorboard matplotlib tqdm opencv-python 
import torch
import torchvision
from torchvision.models.detection import (fcos_resnet50_fpn, retinanet_resnet50_fpn_v2, fasterrcnn_resnet50_fpn_v2)
from torchvision.models.detection.fcos import FCOS_ResNet50_FPN_Weights
from torchvision.models.detection.retinanet import RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_V2_Weights

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from tqdm import tqdm
import json
import os
import glob
import cv2
import warnings
warnings.filterwarnings("ignore")

2025-10-31 19:27:23.619919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761938843.812567      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761938843.872763      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_eval.py
!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_utils.py
!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/engine.py
!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/utils.py
!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/transforms.py

--2025-10-31 19:27:35--  https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_eval.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6447 (6.3K) [text/plain]
Saving to: ‘coco_eval.py’

coco_eval.py        100%[===================>]   6.30K  --.-KB/s    in 0s      

2025-10-31 19:27:35 (68.3 MB/s) - ‘coco_eval.py’ saved [6447/6447]

--2025-10-31 19:27:36--  https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8409 (8.2K) 

In [3]:
!rm -rf /kaggle/working/*

Что-то не работает с фалом convert_coco.py, который есть в датасете, поэтому будет новая функция конвертации из yolo формата в coco для использования pycocotools.

In [4]:
def yolo_to_coco(images_dir, labels_dir, output_json_path, categories, dataset_description="YOLO to COCO converted dataset"):
    if categories is None:
        categories = ["object"]

    coco_output = {"info": {"description": dataset_description},
                            "licenses": [],
                            "images": [],
                            "annotations": [],
                            "categories": [{"id": i + 1, "name": name} for i, name in enumerate(categories)]}

    annotation_id = 1
    image_id = 1

    label_files = sorted(glob.glob(os.path.join(labels_dir, "*.txt")))

    for label_file in label_files:
        base_name = os.path.splitext(os.path.basename(label_file))[0]

        image_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            candidate = os.path.join(images_dir, base_name + ext)
            if os.path.exists(candidate):
                image_path = candidate
                break

        if image_path is None:
            print(f"Изображение для {label_file} не найдено.")
            continue

        with Image.open(image_path) as img:
            width, height = img.size

        coco_output["images"].append({"id": image_id, "file_name": os.path.basename(image_path), "width": width, "height": height})

        with open(label_file, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try: 
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                bbox_width = float(parts[3])
                bbox_height = float(parts[4])
            except ValueError:
                continue

            if class_id < 0 or class_id >= len(categories):
                print(f"Некорректный class_id {class_id} в файле {label_file}.")
                continue
                
            abs_x_center = x_center * width
            abs_y_center = y_center * height
            abs_bbox_width = bbox_width * width
            abs_bbox_height = bbox_height * height

            x_min = abs_x_center - abs_bbox_width / 2
            y_min = abs_y_center - abs_bbox_height / 2

            x_min = max(0.0, x_min)
            y_min = max(0.0, y_min)
            abs_bbox_width = min(abs_bbox_width, width - x_min)
            abs_bbox_height = min(abs_bbox_height, height - y_min)

            if abs_bbox_width <= 0 or abs_bbox_height <= 0:
                continue

            coco_output["annotations"].append({"id": annotation_id, "image_id": image_id, "category_id": class_id + 1,
                                                "bbox": [float(x_min), float(y_min), float(abs_bbox_width), float(abs_bbox_height)],
                                                "area": float(abs_bbox_width * abs_bbox_height), "iscrowd": 0})
            annotation_id += 1
        image_id += 1

    with open(output_json_path, 'w') as f:
        json.dump(coco_output, f, indent=4)

    print(f"Результат конвертации сохранён в {output_json_path}")

Преобразуем информацию из txt файлов для yolo в json файл для pycocotools

In [5]:
classes = ['vehicle', 'bus', 'bicycle', 'pedestrian', 'engine', 'truck', 'tricycle', 'obstacle']
yolo_to_coco(
    images_dir="/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/images/train",
    labels_dir="/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/labels/train",
    output_json_path="/kaggle/working/coco_train.json",
    categories=classes)

Результат конвертации сохранён в /kaggle/working/coco_train.json


In [6]:
yolo_to_coco(
    images_dir="/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/images/val",
    labels_dir="/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/labels/val",
    output_json_path="/kaggle/working/coco_val.json",
    categories=classes)

Результат конвертации сохранён в /kaggle/working/coco_val.json


In [7]:
import os
from PIL import Image
import torch
from pycocotools.coco import COCO
import torchvision.transforms as T


class COCODataset(torch.utils.data.Dataset):
    def __init__(self, root: str, ann_file: str, transforms=None):
        self.root = root
        self.transforms = transforms
        self.coco = COCO(ann_file)
        self.ids = list(sorted(self.coco.imgs.keys()))

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(img_id)[0]
        path = img_info["file_name"]
        img = Image.open(os.path.join(self.root, path)).convert("RGB")

        boxes = []
        labels = []
        for ann in anns:
            if ann.get('iscrowd', 0) == 1:
                continue
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

        if len(boxes) == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor(img_id)}

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.ids)


def get_transform(train: bool = True):
    transforms = []
    transforms.append(T.ToTensor()) 
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    return T.Compose(transforms)

In [8]:
working_root = "/kaggle/input/intersection-flow-5k/Intersection-Flow-5K"
train_img_dir = os.path.join(working_root, "images", "train")
val_img_dir = os.path.join(working_root, "images", "val")
train_ann = "/kaggle/working/coco_train.json"
val_ann = "/kaggle/working/coco_val.json"

train_dataset = COCODataset(train_img_dir, train_ann, transforms=get_transform(train=True))
val_dataset = COCODataset(val_img_dir, val_ann, transforms=get_transform(train=False))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))

coco_gt = COCO(val_ann)

print(f"Train: {len(train_dataset)} изображений")
print(f"Val: {len(val_dataset)} изображений")

loading annotations into memory...
Done (t=2.10s)
creating index...
index created!
loading annotations into memory...
Done (t=0.18s)
creating index...
index created!
loading annotations into memory...
Done (t=0.12s)
creating index...
index created!
Train: 5483 изображений
Val: 722 изображений


Напишем функции для тренировки и оценки качества модели.

In [9]:
def train_one_epoch(model, optimizer, data_loader, device, epoch, writer, tb_step):
    model.train()
    total_loss = 0
    pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}")
    for batch_idx, (images, targets) in enumerate(pbar):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        writer.add_scalar('Loss/train_step', losses.item(), tb_step)
        tb_step += 1

    avg_loss = total_loss / len(data_loader)
    writer.add_scalar('Loss/train_epoch', avg_loss, epoch)
    return tb_step

def evaluate_model(model, data_loader, device, coco_gt, writer, epoch):
    model.eval()
    results = []
    with torch.no_grad():
        pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}")
        for batch_idx, (images, targets) in enumerate(pbar):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, output in enumerate(outputs):
                boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()
                labels = output['labels'].cpu().numpy()
                img_id = targets[i]["image_id"].item()

                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = box
                    w, h = x2 - x1, y2 - y1
                    if w > 0 and h > 0 and score > 0.01:
                        results.append({"image_id": int(img_id), "category_id": int(label), 
                                        "bbox": [float(x1), float(y1), float(w), float(h)], "score": float(score)})

    res_file = f"/kaggle/working/val_results_{epoch}.json"
    with open(res_file, 'w') as f:
        json.dump(results, f)

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    writer.add_scalar('mAP/val', coco_eval.stats[0], epoch)
    writer.add_scalar('mAP@0.5/val', coco_eval.stats[1], epoch)

    return coco_eval.stats[0], results
    

In [10]:
def visualize_detections(model, data_loader, device, writer, epoch, class_names, max_images=3):
    model.eval()
    images_logged = 0
    with torch.no_grad():
        for images, targets in data_loader:
            if images_logged >= max_images:
                break
            img = images[0].to(device)
            pred = model([img])[0]

            img_np = img.cpu().permute(1, 2, 0).numpy()
            img_np = (img_np * 255).astype(np.uint8).copy()

            boxes = pred['boxes'].cpu().numpy()
            scores = pred['scores'].cpu().numpy()
            labels = pred['labels'].cpu().numpy()

            for box, score, label in zip(boxes, scores, labels):
                if score > 0.5 and images_logged < max_images:
                    x1, y1, x2, y2 = map(int, box)
                    cls_name = class_names.get(label, str(label))
                    cv2.rectangle(img_np, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(img_np, f"{cls_name}: {score:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

            writer.add_image(f'Val/Detection_{images_logged}', img_np, epoch, dataformats='HWC')
            images_logged += 1

Обозначим конфигурации моделей и заменим последние слои.

In [11]:
num_object_classes = 8
class_names = {i: name for i, name in enumerate([
    'vehicle', 'bus', 'bicycle', 'pedestrian', 
    'engine', 'truck', 'tricycle', 'obstacle'
])}

models_config = {
    "fcos": {
        "model": fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.COCO_V1),
        "num_classes": num_object_classes + 1
    },
    "retinanet": {
        "model": retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1),
        "num_classes": num_object_classes 
    },
    "fasterrcnn": {
        "model": fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1),
        "num_classes": num_object_classes 
    }
}

for name, cfg in models_config.items():
    if name == "fcos":
        head = cfg["model"].head.classification_head
        in_channels = head.cls_logits.in_channels
        out_channels = head.cls_logits.out_channels 
        kernel_size = head.cls_logits.kernel_size 
        stride = head.cls_logits.stride
    
        new_cls_logits = torch.nn.Conv2d(
            in_channels=in_channels,
            out_channels=cfg["num_classes"],
            kernel_size=kernel_size,
            stride=stride,
            padding=1
        )
    
        torch.nn.init.normal_(new_cls_logits.weight, std=0.01)
        torch.nn.init.constant_(new_cls_logits.bias, -4.595)

        cfg["model"].head.classification_head.cls_logits = new_cls_logits

        cfg["model"].head.classification_head.num_classes = cfg["num_classes"]
    elif name == "retinanet":
        cfg["model"].head.classification_head.num_classes = cfg["num_classes"]
        torch.nn.init.normal_(cfg["model"].head.classification_head.cls_logits.weight, std=0.01)
        torch.nn.init.constant_(cfg["model"].head.classification_head.cls_logits.bias, -4.595)
    elif name == "fasterrcnn":
        in_features = cfg["model"].roi_heads.box_predictor.cls_score.in_features
        cfg["model"].roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, cfg["num_classes"])

Downloading: "https://download.pytorch.org/models/fcos_resnet50_fpn_coco-99b0c9b7.pth" to /root/.cache/torch/hub/checkpoints/fcos_resnet50_fpn_coco-99b0c9b7.pth
100%|██████████| 124M/124M [00:01<00:00, 93.3MB/s] 
Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth
100%|██████████| 146M/146M [00:01<00:00, 95.0MB/s] 
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
100%|██████████| 167M/167M [00:01<00:00, 88.4MB/s] 


In [12]:
import gc
gc.collect()
torch.cuda.empty_cache()

Следующая ячейка выполнилась только для модели fcos, поэтому я ее закомментировала, чтобы случайно снова не запустить.

In [14]:
"""
import gc
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
results = {}
for name, cfg in models_config.items():
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Обучение модели: {name.upper()}")
    
    model = cfg["model"].to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
    
    writer = SummaryWriter(log_dir=f"/kaggle/working/runs/{name}")
    tb_step = 0
    best_map = 0.0

    for epoch in range(5):
        tb_step = train_one_epoch(model, optimizer, train_loader, device, epoch, writer, tb_step)
        
        if epoch % 2 == 0 or epoch == 4:
            val_map, _ = evaluate_model(model, val_loader, device, coco_gt, writer, epoch)
            visualize_detections(model, val_loader, device, writer, epoch, class_names)
            if val_map > best_map:
                best_map = val_map
                torch.save(model.state_dict(), f"/kaggle/working/{name}_best.pth")
                print(f"Сохранена лучшая модель: mAP = {best_map:.4f}")

    results[name] = best_map
    writer.close()
    print(f"{name}: лучший mAP = {best_map:.4f}")
"""

Обучение модели: FCOS


Epoch 1: 100%|██████████| 181/181 [00:56<00:00,  3.20it/s]


Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=21.82s).
Accumulating evaluation results...
DONE (t=0.82s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.217
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.339
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.231
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.064
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.215
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.386
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.142
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.297
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.348
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Epoch 3: 100%|██████████| 181/181 [00:56<00:00,  3.23it/s]


Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=21.14s).
Accumulating evaluation results...
DONE (t=0.82s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.327
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.485
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.352
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.098
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.333
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.520
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.209
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.378
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.434
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Epoch 5: 100%|██████████| 181/181 [00:56<00:00,  3.20it/s]


Loading and preparing results...
DONE (t=0.80s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=21.44s).
Accumulating evaluation results...
DONE (t=0.79s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.359
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.528
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.391
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.143
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.373
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.544
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.234
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.406
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.465
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Epoch 1:   0%|          | 0/1371 [00:01<?, ?it/s]


RuntimeError: shape '[4, -1, 8, 96, 168]' is invalid for input of size 52835328

Значение метрики получилось примерно как у yolo11n в каком-то из экспериментов в части 3.

In [13]:
num_object_classes = 8
class_names = {i: name for i, name in enumerate([
    'vehicle', 'bus', 'bicycle', 'pedestrian', 
    'engine', 'truck', 'tricycle', 'obstacle'
])}

models_config_new = {
    "fasterrcnn": {
        "model": fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1),
        "num_classes": num_object_classes 
    }
}

for name, cfg in models_config_new.items():
    if name == "fasterrcnn":
        in_features = cfg["model"].roi_heads.box_predictor.cls_score.in_features
        cfg["model"].roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
            in_features, cfg["num_classes"]
        )

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for name, cfg in models_config_new.items():
    
    print(f"Обучение модели: {name.upper()}")
    
    model = cfg["model"].to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
    
    writer = SummaryWriter(log_dir=f"/kaggle/working/runs/{name}")
    tb_step = 0
    best_map = 0.0

    for epoch in range(5):
        tb_step = train_one_epoch(model, optimizer, train_loader, device, epoch, writer, tb_step)
        
        if epoch % 2 == 0:
            val_map, _ = evaluate_model(model, val_loader, device, coco_gt, writer, epoch)
            visualize_detections(model, val_loader, device, writer, epoch, class_names)
            if val_map > best_map:
                best_map = val_map
                torch.save(model.state_dict(), f"/kaggle/working/{name}_best.pth")
                print(f"Сохранена лучшая модель: mAP = {best_map:.4f}")

    results[name] = best_map
    writer.close()
    print(f"{name}: лучший mAP = {best_map:.4f}")

Обучение модели: FASTERRCNN


Epoch 1:   0%|          | 0/1371 [00:02<?, ?it/s]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Не поняла, что за ошибка. Если запустить сессию без гпу, а потом снова с гпу, то ошибка все равно остается. В общем, не получилось справиться с этим заданием. Во-первых, очень долго обучается модель, даже 1 эпоха занимает кучу времени, ставить размер батча больше 4 не очень, так как вылетает по памяти. Сколько по времени обучать всю модель, а не только последние слои, я даже боюсь представить.